In [40]:
import sys
sys.path.append('..')
from osp import *

In [41]:
df_meta = get_corpus_metadata()

In [42]:
df_preds = pd.read_pickle('../data/raw/df_preds3.pkl.gz')


KeyboardInterrupt: 

In [ ]:
df_preds_cv = df_preds.query('predict_type=="cv"')
df_preds_cv['same_period'] = [x.split()[0] == x.split()[-2] for x in df_preds_cv['comparison']]
df_preds_cv['period_model'] = df_preds_cv['comparison'].str.split().str[0]
df_preds_cv = df_preds_cv[df_preds_cv.same_period]
df_preds_cv

,true_label,pred_label,prob_1900-1925 Philosophy,prob_1925-1950 Philosophy,test_label,confidence,correct,accuracy,support,run,...,prob_1950-1975 Literature,prob_1975-2000 Literature,prob_2000-2025 Literature,prob_1900-1925 Other,prob_1925-1950 Other,prob_1950-1975 Other,prob_1975-2000 Other,prob_2000-2025 Other,same_period,period_model
id,,,,,,,,,,,,,,,,,,,,,
lit/432896__01,1900-1925 Literature,1900-1925 Literature,0.000094,NaN,1900-1925 Literature / 1900-1925 Philosophy,0.999906,True,0.843243,370,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,1900-1925
lit/457039__01,1900-1925 Literature,1900-1925 Literature,0.000353,NaN,1900-1925 Literature / 1900-1925 Philosophy,0.999647,True,0.843243,370,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,1900-1925
lit/456616__03,1900-1925 Literature,1900-1925 Literature,0.040258,NaN,1900-1925 Literature / 1900-1925 Philosophy,0.959742,True,0.843243,370,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,1900-1925
lit/433191__01,1900-1925 Literature,1900-1925 Literature,0.000149,NaN,1900-1925 Literature / 1900-1925 Philosophy,0.999851,True,0.843243,370,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,1900-1925
lit/3713552__01,1900-1925 Literature,1900-1925 Literature,0.001884,NaN,1900-1925 Literature / 1900-1925 Philosophy,0.998116,True,0.843243,370,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,1900-1925
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
other/10.2307/26888705__04,2000-2025 Other,2000-2025 Literature,NaN,NaN,2000-2025 Literature / 2000-2025 Other,0.980330,False,0.801370,146,9,...,NaN,NaN,0.980330,NaN,NaN,NaN,NaN,0.019670,True,2000-2025
other/10.2307/27290935__01,2000-2025 Other,2000-2025 Other,NaN,NaN,2000-2025 Literature / 2000-2025 Other,0.963393,True,0.801370,146,9,...,NaN,NaN,0.036607,NaN,NaN,NaN,NaN,0.963393,True,2000-2025
other/10.2307/24487550__01,2000-2025 Other,2000-2025 Other,NaN,NaN,2000-2025 Literature / 2000-2025 Other,0.978502,True,0.801370,146,9,...,NaN,NaN,0.021498,NaN,NaN,NaN,NaN,0.978502,True,2000-2025


In [ ]:
gby=['period_model','comparison','run']
g2acc = defaultdict(list)
for g,gdf in df_preds_cv.groupby(gby):
    cmp = g[1]
    g2acc[cmp] += [gdf['correct'].mean()]

In [ ]:
g2acc2 = {}
for cmp,acc in g2acc.items():
    g2acc2[cmp] = f'{np.mean(acc)*100:.1f}% PLUSMINUS {np.std(acc)*100:.1f}%'
g2acc2

{'1900-1925 Literature vs 1900-1925 Other': '79.2% PLUSMINUS 2.1%',
 '1900-1925 Philosophy vs 1900-1925 Literature': '85.4% PLUSMINUS 1.0%',
 '1900-1925 Philosophy vs 1900-1925 Other': '87.8% PLUSMINUS 1.0%',
 '1925-1950 Literature vs 1925-1950 Other': '83.5% PLUSMINUS 1.3%',
 '1925-1950 Philosophy vs 1925-1950 Literature': '87.9% PLUSMINUS 1.4%',
 '1925-1950 Philosophy vs 1925-1950 Other': '88.8% PLUSMINUS 1.6%',
 '1950-1975 Literature vs 1950-1975 Other': '79.8% PLUSMINUS 3.3%',
 '1950-1975 Philosophy vs 1950-1975 Literature': '88.2% PLUSMINUS 0.6%',
 '1950-1975 Philosophy vs 1950-1975 Other': '84.7% PLUSMINUS 2.4%',
 '1975-2000 Literature vs 1975-2000 Other': '78.5% PLUSMINUS 3.8%',
 '1975-2000 Philosophy vs 1975-2000 Literature': '88.7% PLUSMINUS 0.5%',
 '1975-2000 Philosophy vs 1975-2000 Other': '86.9% PLUSMINUS 3.6%',
 '2000-2025 Literature vs 2000-2025 Other': '80.7% PLUSMINUS 3.3%',
 '2000-2025 Philosophy vs 2000-2025 Literature': '92.0% PLUSMINUS 0.5%',
 '2000-2025 Philosophy 

In [ ]:
ld = []
for cmp,acc in g2acc2.items():
    ld.append({
        'period':cmp.split()[0],
        'disciplines':cmp.split()[1] + ' vs ' + cmp.split()[-1],
        'acc':acc,
    })
ldf = pd.DataFrame(ld)
odf = ldf.pivot(index='period',columns='disciplines',values='acc')
odf = odf[['Philosophy vs Literature', 'Philosophy vs Other', 'Literature vs Other']]
odf

disciplines,Philosophy vs Literature,Philosophy vs Other,Literature vs Other
period,,,
1900-1925,85.4% PLUSMINUS 1.0%,87.8% PLUSMINUS 1.0%,79.2% PLUSMINUS 2.1%
1925-1950,87.9% PLUSMINUS 1.4%,88.8% PLUSMINUS 1.6%,83.5% PLUSMINUS 1.3%
1950-1975,88.2% PLUSMINUS 0.6%,84.7% PLUSMINUS 2.4%,79.8% PLUSMINUS 3.3%
1975-2000,88.7% PLUSMINUS 0.5%,86.9% PLUSMINUS 3.6%,78.5% PLUSMINUS 3.8%
2000-2025,92.0% PLUSMINUS 0.5%,84.3% PLUSMINUS 4.0%,80.7% PLUSMINUS 3.3%


In [ ]:
out=df_to_latex_table(odf.reset_index(), caption="""
The accuracy of the model for each period and discipline.
""".strip(), label="table:cv_acc")
out = out.replace('PLUSMINUS', '$\pm$')

In [ ]:
with open('../../../Dropbox/Prof/Articles/OSP/tables/table.cv_acc.tex', 'w') as f:
    f.write(out)